In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType


In [0]:
stations = [
    {"name": "NEW DELHI",                  "lat": 28.6419, "lon": 77.2197},
    {"name": "CHENNAI CENTRAL",            "lat": 13.0827, "lon": 80.2707},
    {"name": "HOWRAH JN",                  "lat": 22.5839, "lon": 88.3424},
    {"name": "VIJAYAWADA JN",              "lat": 16.5167, "lon": 80.6167},
    {"name": "KALYAN JN",                  "lat": 19.2437, "lon": 73.1355},
    {"name": "PT DEEN DAYAL UPADHYAY JN", "lat": 25.2130, "lon": 83.0451},
    {"name": "SALEM JN",                   "lat": 11.6643, "lon": 78.1460},
    {"name": "AGRA CANTT",                 "lat": 27.1591, "lon": 78.0089},
    {"name": "ARAKKONAM",                  "lat": 13.0786, "lon": 79.6717},
    {"name": "KATPADI JN",                 "lat": 12.9247, "lon": 79.1386},
]

# Date range — monsoon season
START_DATE = "2025-07-01"
END_DATE   = "2025-09-30"

In [0]:
def fetch_weather(station):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":        station["lat"],
        "longitude":       station["lon"],
        "start_date":      START_DATE,
        "end_date":        END_DATE,
        "daily":           "precipitation_sum,temperature_2m_max",
        "timezone":        "Asia/Kolkata"
    }

    response = requests.get(url, params=params, timeout=30)

    # Check the request actually succeeded
    if response.status_code != 200:
        print(f"ERROR: {station['name']} returned status {response.status_code}")
        return []

    data = response.json()

    # Check the expected keys exist in the response
    if "daily" not in data:
        print(f"ERROR: No daily data returned for {station['name']}")
        return []

    dates  = data["daily"]["time"]
    precip = data["daily"]["precipitation_sum"]
    temp   = data["daily"]["temperature_2m_max"]

    rows = []
    for i, date in enumerate(dates):
        rows.append({
            "station_name":       station["name"],
            "latitude":           station["lat"],
            "longitude":          station["lon"],
            "date":               date,
            "precipitation_mm":   precip[i],
            "temp_max_c":         temp[i]
        })

    return rows

In [0]:
all_rows = []

for station in stations:
    print(f"Fetching: {station['name']}...")
    rows = fetch_weather(station)
    all_rows.extend(rows)
    print(f"  Got {len(rows)} rows")

print(f"\nTotal rows collected: {len(all_rows)}")

In [0]:
pandas_df = pd.DataFrame(all_rows)

# Quick inspection
print(f"Shape: {pandas_df.shape}")
print(f"\nColumn names: {list(pandas_df.columns)}")
print(f"\nNull counts:\n{pandas_df.isnull().sum()}")
print(f"\nSample rows:")
pandas_df.head(10)

In [0]:
spark_df = spark.createDataFrame(pandas_df)

spark_df.printSchema()

In [0]:
spark_df = spark_df.withColumn(
    "date",
    F.to_date("date", "yyyy-MM-dd")
)

spark_df.printSchema()

In [0]:
spark_df = spark_df.fillna({
    "precipitation_mm": 0.0,
    "temp_max_c":       0.0
})

In [0]:
print(f"Row count: {spark_df.count()}")
print(f"\nSchema:")
spark_df.printSchema()
print(f"\nSample rows:")
spark_df.show(10, truncate=False)
print(f"\nNull counts:")
spark_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in spark_df.columns
]).show()

In [0]:
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("weather_history")
    

In [0]:
# Check row count
spark.sql("SELECT COUNT(*) as total_rows FROM weather_history").show()

# Check one row per station — confirms all 10 stations are present
spark.sql("""
    SELECT station_name, COUNT(*) as row_count
    FROM weather_history
    GROUP BY station_name
    ORDER BY station_name
""").show(truncate=False)

# Check date range is correct
spark.sql("""
    SELECT 
        MIN(date) as earliest_date,
        MAX(date) as latest_date
    FROM weather_history
""").show()

# Check a sample of actual data
spark.sql("""
    SELECT *
    FROM weather_history
    WHERE station_name = 'CHENNAI CENTRAL'
    ORDER BY date
    LIMIT 5
""").show(truncate=False)

In [0]:
# Confirm it's a Delta table
spark.sql("DESCRIBE DETAIL weather_history").select("format", "location", "numFiles").show(truncate=False)

In [0]:
spark.sql("DESCRIBE HISTORY weather_history").select("version", "timestamp", "operation").show(truncate=False)